In [1]:
## Ticket Classification
#TF-IDF + Logistic Regression

In [3]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

pd.set_option('display.max_colwidth', 150)


In [10]:
DATA_PATH = '../dataset/twcs.csv'

df = pd.read_csv(DATA_PATH)
df.shape

(2811774, 7)

In [11]:
df.head()

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,1,sprintcare,False,Tue Oct 31 22:10:47 +0000 2017,@115712 I understand. I would like to assist you. We would need to get you into a private secured link to further assist.,2,3.0
1,2,115712,True,Tue Oct 31 22:11:45 +0000 2017,@sprintcare and how do you propose we do that,NaN,1.0
2,3,115712,True,Tue Oct 31 22:08:27 +0000 2017,@sprintcare I have sent several private messages and no one is responding as usual,1,4.0
3,4,sprintcare,False,Tue Oct 31 21:54:49 +0000 2017,@115712 Please send us a Private Message so that we can further assist you. Just click ‘Message’ at the top of your profile.,3,5.0
4,5,115712,True,Tue Oct 31 21:49:35 +0000 2017,@sprintcare I did.,4,6.0


In [12]:
# only care about customer messages not company replies
df = df[df['inbound'] == True].copy()
len(df)

1537843

In [13]:
def clean(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'@\w+', '', text)      # removing @mentions
    text = re.sub(r'http\S+', '', text)   # removing links
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['text_clean'] = df['text'].apply(clean)

In [14]:
df['word_count'] = df['text_clean'].str.split().str.len()
df['word_count'].describe()

count    1.537843e+06
mean     1.735079e+01
std      1.059808e+01
min      0.000000e+00
25%      9.000000e+00
50%      1.700000e+01
75%      2.300000e+01
max      1.370000e+02
Name: word_count, dtype: float64

In [15]:
# dropping super short and long ones, mostly noise
df = df[(df['word_count'] >= 3) & (df['word_count'] <= 200)]
len(df)

1446925

In [16]:
print(df.isnull().sum())

tweet_id                        0
author_id                       0
inbound                         0
created_at                      0
text                            0
response_tweet_id          201122
in_response_to_tweet_id    771770
text_clean                      0
word_count                      0
dtype: int64


In [17]:
## Labelling
# No category column in this dataset so labeling based on keywords for now. Categories picked after reading through ~50 random tickets manually. Will probably need to tweak these once I see how the model does.

In [18]:
categories = {
    'Billing': ['charge', 'charged', 'refund', 'invoice', 'payment', 'billing', 'subscription', 'overcharge'],
    'Technical': ['not working', 'bug', 'error', 'crash', "won't load", 'broken', 'glitch', "can't login", 'app keeps'],
    'Shipping': ['delivery', 'shipped', 'shipping', 'package', 'tracking', "hasn't arrived", 'delayed', 'order status'],
    'Account': ['account', 'password', 'reset', 'locked out', 'verify', 'login', 'sign in'],
    'General': ['how do i', 'how can i', 'question about', 'wondering if']
}

In [19]:
def get_category(text):
    text = text.lower()
    for cat, keywords in categories.items():
        for kw in keywords:
            if kw in text:
                return cat
    return None

df['category'] = df['text_clean'].apply(get_category)
df['category'].value_counts(dropna=False)

category
NaN          1197428
Billing        69167
Shipping       63598
Account        52413
Technical      48927
General        15392
Name: count, dtype: int64

In [20]:
# dropping the rows having no match in defined categories
labeled_df = df[df['category'].notna()].copy()
len(labeled_df)

249497

In [21]:
# Checking if categories fall right
for cat in labeled_df['category'].unique():
    print('===', cat, '===')
    print(labeled_df[labeled_df['category'] == cat]['text_clean'].sample(3, random_state=1).values)
    print()

=== Account ===
<StringArray>
['Telling me what you doing won’t help. I want to know why you take funds, put them back in account, take them out again. #WellsFargo',
      'ever since the iOS update, I have to reset my phone EVERY time I plug it into my car for it to play music. Never did it before',
                            'It was a local account. Brand new computer. The computer had been on sleep mode. I had to reset Windows.']
Length: 3, dtype: str

=== Technical ===
<StringArray>
[                                                                                 'Thats for my shipping address. That error is from the app. That error dont let me preorder last year and want to fix',
 'LOVE that the cd gift I ordered from was damaged during shipping so they notified me &amp; sent a replacement... but put the replacement in an envelope, not a BOX, so it’s broken again! Awesome! 👌🏼',
                                                                                                       

In [23]:
# Pass the data as NumPy arrays by adding .values
X_train, X_test, y_train, y_test = train_test_split(
    labeled_df['text_clean'].values, 
    labeled_df['category'].values,
    test_size=0.2, 
    random_state=42, 
    stratify=labeled_df['category'].values
)

In [24]:
# TF-IDF
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

X_train_tfidf.shape

(199597, 5000)

In [25]:
# Logistic Regression
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_tfidf, y_train)

preds = clf.predict(X_test_tfidf)
print(accuracy_score(y_test, preds))

0.9806613226452906


In [26]:
print(classification_report(y_test, preds))

              precision    recall  f1-score   support

     Account       0.98      0.99      0.98     10483
     Billing       1.00      0.98      0.99     13834
     General       0.88      0.97      0.92      3078
    Shipping       0.99      0.99      0.99     12720
   Technical       0.98      0.96      0.97      9785

    accuracy                           0.98     49900
   macro avg       0.97      0.98      0.97     49900
weighted avg       0.98      0.98      0.98     49900



In [27]:
# quick look at a few wrong predictions to see what's going on
wrong = X_test[preds != y_test.values]
wrong.head(10)

AttributeError: 'StringArray' object has no attribute 'values'

In [29]:
import os
import pickle

os.makedirs("saved_models", exist_ok=True)

with open("saved_models/ticket_classifier.pkl", "wb") as f:
    pickle.dump(clf, f)

with open("saved_models/tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf, f)

labeled_df.to_csv("../dataset/labeled_tickets.csv", index=False)